# ADS queue reset helper

This Positron script scans local CAMS/ADS state files, extracts recent ADS job identifiers, checks their current server-side status, and cancels only jobs that are still active on the ADS server.

It is mainly useful after interrupted CAMS download runs, failed API keys, disabled keys, or aborted Positron/Python sessions. The goal is to prevent self-queuing, where old `accepted`, `queued`, or `running` jobs remain on the ADS server and continue occupying your request slab even though the local script was stopped.

## What the script does

1. Reads local `.jsonl`, `.log`, and `.csv` files from the configured CAMS state folders.
2. Extracts ADS job identifiers.
3. Uses the stored `key_index` when available, otherwise tests the job against all configured ADS keys.
4. Queries the ADS job endpoint:

   `https://ads.atmosphere.copernicus.eu/api/retrieve/v1/jobs/<job_id>`

5. Cancels only jobs with active statuses:

   `accepted`, `submitted`, `queued`, `running`, `created`, `pending`

6. Leaves terminal jobs untouched:

   `successful`, `failed`, `rejected`, `dismissed`, `deleted`, `not_found`

7. Writes a CSV audit table to:

   `D:/CAMS_ads_reset/ads_active_jobs_cancelled_parallel.csv`

## Server health / status check

Before launching large batches, check the Copernicus Atmosphere Data Store service status here:

https://status.ecmwf.int/

Also check the ADS web interface if the API behaves unusually:

https://ads.atmosphere.copernicus.eu/live

## Why this matters

Interrupted ADS jobs may remain active remotely even when the local Python process has stopped. If many old jobs are still `accepted`, `queued`, or `running`, new requests can stall behind your own abandoned jobs. This script clears those active server-side jobs without deleting local files.

## Safety behaviour

The script is conservative:

- Local files are read only.
- Nothing is deleted from disk.
- Completed, failed, rejected, dismissed, deleted, or inaccessible jobs are not cancelled.
- Only active remote ADS jobs are cancelled.
- Every checked job is recorded in the output CSV.

## Main settings

```python
MAX_JOB_IDS = 1000
N_WORKERS = 15
HTTP_TIMEOUT = 15


In [ ]:
#use this to reset your ads api slab
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import re
import time
import csv
import requests
import threading

ADS_URL = "https://ads.atmosphere.copernicus.eu/api"
HTTP_TIMEOUT = 15
MAX_JOB_IDS = 1000
N_WORKERS = 15

ADS_KEYS = [
    "156656bf-4346-4228-8ea6-a47b8b807223",
    "1e0d795d-e02a-4ebc-81f9-0fb9c07f85a9",
    "1e912241-e383-49bc-a605-4f3155b60ad6",
    "9f9d3677-2705-4764-832b-c5d9bc917eb0",
    "9b7c19b2-d4c7-4ab6-9aef-33d57355862a",
    "2d7328f2-7e18-4fb4-ad55-2ca4605eb1c4",
    "22dec1aa-ecf2-4b87-9641-b80361fdba06",
    "d952344a-3e13-45a6-a7cd-4a7b8e220a79",
    "b6a8b25e-6d7f-46f8-80f6-cf0e37a2719b",
    "40b9f769-d2f6-4369-9b1f-15071345c640",
    "71c70660-fc78-41ce-836b-562ad7f3e1bc",
    "27446185-27c3-4c5d-b178-debf0401be69",
    "54964f95-2780-4c66-b9ef-43e8443d1335",
    "9fd1440b-d970-438c-a31d-1b68807111df",
    "1058bf2f-507b-409e-b072-e36c65f5415d",
]

SEARCH_DIRS = [
    Path(r"D:/CAMS_modellevel_aerosol_monthly/_state"),
    Path(r"D:/CAMS_modellevel_aerosol_monthly"),
    Path(r"D:/CAMS_key_test_ireland_20210403_18UTC"),
]

OUTDIR = Path(r"D:/CAMS_ads_reset")
OUTDIR.mkdir(parents=True, exist_ok=True)
OUTCSV = OUTDIR / "ads_active_jobs_cancelled_parallel.csv"

JOB_RE = re.compile(r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b", re.I)

ACTIVE = {"accepted", "submitted", "queued", "running", "created", "pending"}
TERMINAL = {"successful", "failed", "rejected", "dismissed", "deleted", "not_found"}
WRITE_LOCK = threading.Lock()

def headers(key):
    return {"PRIVATE-TOKEN": key}

def job_url(job_id):
    return f"{ADS_URL}/retrieve/v1/jobs/{job_id}"

def status_of(session, key, job_id):
    r = session.get(job_url(job_id), headers=headers(key), timeout=HTTP_TIMEOUT)
    txt = r.text[:1000]

    if r.status_code == 404:
        return "not_found", txt
    if r.status_code in (401, 403):
        return "auth_error", txt
    if r.status_code >= 400:
        return f"http_{r.status_code}", txt

    body = r.json()
    return str(body.get("status", "")).lower(), json.dumps(body)[:1000]

def cancel_job(session, key, job_id):
    r = session.delete(job_url(job_id), headers=headers(key), timeout=HTTP_TIMEOUT)
    return r.status_code, r.text[:1000]

def local_files():
    files = []
    for d in SEARCH_DIRS:
        if not d.exists():
            continue
        files.extend(d.glob("*.jsonl"))
        files.extend(d.glob("*.log"))
        files.extend(d.glob("*.csv"))
    return sorted(files, key=lambda p: p.stat().st_mtime, reverse=True)

def extract_jobs_with_key_index():
    jobs = {}

    for p in local_files():
        try:
            txt = p.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue

        if p.suffix.lower() == ".jsonl":
            for line in txt.splitlines():
                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                job_id = obj.get("job_id")
                key_index = obj.get("key_index")
                event = obj.get("event", "")
                ts = p.stat().st_mtime

                if job_id and JOB_RE.fullmatch(str(job_id)) and key_index is not None:
                    job_id = str(job_id).lower()
                    old = jobs.get(job_id)
                    if old is None or ts > old["mtime"]:
                        jobs[job_id] = {
                            "job_id": job_id,
                            "key_index": int(key_index),
                            "source": str(p),
                            "event": event,
                            "mtime": ts,
                        }

        for jid in JOB_RE.findall(txt):
            jid = jid.lower()
            if jid not in jobs:
                jobs[jid] = {
                    "job_id": jid,
                    "key_index": None,
                    "source": str(p),
                    "event": "",
                    "mtime": p.stat().st_mtime,
                }

    return sorted(jobs.values(), key=lambda x: x["mtime"], reverse=True)

def test_and_cancel(rec):
    job_id = rec["job_id"]
    known_key = rec["key_index"]
    session = requests.Session()

    if known_key is not None and 0 <= known_key < len(ADS_KEYS):
        key_order = [known_key]
    else:
        key_order = list(range(len(ADS_KEYS)))

    last_auth = None

    for i in key_order:
        key = ADS_KEYS[i]

        try:
            st, msg = status_of(session, key, job_id)
        except Exception as e:
            return {
                "job_id": job_id,
                "key_index": i,
                "key_prefix": key[:8],
                "initial_status": "local_error",
                "delete_http": "",
                "final_status": "",
                "source": rec["source"],
                "message": repr(e)[:1000],
            }

        if st == "not_found":
            continue

        if st == "auth_error":
            last_auth = {
                "job_id": job_id,
                "key_index": i,
                "key_prefix": key[:8],
                "initial_status": st,
                "delete_http": "",
                "final_status": st,
                "source": rec["source"],
                "message": msg,
            }
            continue

        if st in ACTIVE:
            try:
                code, dmsg = cancel_job(session, key, job_id)
                time.sleep(0.1)
                st2, msg2 = status_of(session, key, job_id)
            except Exception as e:
                return {
                    "job_id": job_id,
                    "key_index": i,
                    "key_prefix": key[:8],
                    "initial_status": st,
                    "delete_http": "delete_error",
                    "final_status": "",
                    "source": rec["source"],
                    "message": repr(e)[:1000],
                }

            return {
                "job_id": job_id,
                "key_index": i,
                "key_prefix": key[:8],
                "initial_status": st,
                "delete_http": code,
                "final_status": st2,
                "source": rec["source"],
                "message": (dmsg or msg2)[:1000],
            }

        return {
            "job_id": job_id,
            "key_index": i,
            "key_prefix": key[:8],
            "initial_status": st,
            "delete_http": "",
            "final_status": st,
            "source": rec["source"],
            "message": msg[:1000],
        }

    if last_auth is not None and known_key is not None:
        return last_auth

    return {
        "job_id": job_id,
        "key_index": "",
        "key_prefix": "",
        "initial_status": "not_accessible_with_any_key",
        "delete_http": "",
        "final_status": "",
        "source": rec["source"],
        "message": "",
    }

def write_csv(rows):
    with WRITE_LOCK:
        with open(OUTCSV, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(
                f,
                fieldnames=[
                    "job_id",
                    "key_index",
                    "key_prefix",
                    "initial_status",
                    "delete_http",
                    "final_status",
                    "source",
                    "message",
                ],
            )
            w.writeheader()
            w.writerows(rows)

jobs = extract_jobs_with_key_index()[:MAX_JOB_IDS]
rows = []

print("Local files are read only. Nothing local will be deleted.")
print(f"Testing newest {len(jobs)} local ADS job IDs.")
print(f"Parallel workers: {N_WORKERS}")
print("Only active server jobs will be cancelled.")
print(f"CSV summary: {OUTCSV}")

t0 = time.time()

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(test_and_cancel, rec): rec for rec in jobs}

    for n, fut in enumerate(as_completed(futures), 1):
        rec = futures[fut]

        try:
            row = fut.result()
        except Exception as e:
            row = {
                "job_id": rec["job_id"],
                "key_index": "",
                "key_prefix": "",
                "initial_status": "worker_crash",
                "delete_http": "",
                "final_status": "",
                "source": rec["source"],
                "message": repr(e)[:1000],
            }

        rows.append(row)

        action = ""
        if row["delete_http"] != "":
            action = f" DELETE={row['delete_http']} final={row['final_status']}"

        print(
            f"{n}/{len(jobs)} "
            f"{row['job_id'][:8]} "
            f"key={row['key_index']} "
            f"{row['initial_status']}"
            f"{action}"
        )

        if n % 20 == 0 or n == len(jobs):
            write_csv(rows)

write_csv(rows)

summary = {}
for r in rows:
    k = r["initial_status"]
    summary[k] = summary.get(k, 0) + 1

print("\nSummary")
print("=" * 80)

for k, v in sorted(summary.items()):
    print(f"{k}: {v}")

print(f"\nElapsed: {time.time() - t0:.1f}s")
print(f"Wrote: {OUTCSV}")

Local files are read only. Nothing local will be deleted.
Testing newest 1000 local ADS job IDs.
Parallel workers: 15
Only active server jobs will be cancelled.
CSV summary: D:\CAMS_ads_reset\ads_active_jobs_cancelled_parallel.csv
1/1000 0f2b30a2 key= not_accessible_with_any_key
2/1000 9924c3fa key= not_accessible_with_any_key
3/1000 e874cbd0 key= not_accessible_with_any_key
4/1000 b145e583 key= not_accessible_with_any_key
5/1000 260e9452 key= not_accessible_with_any_key
6/1000 94e6cdff key= not_accessible_with_any_key
7/1000 de960a2e key= not_accessible_with_any_key
8/1000 9ac113b9 key= not_accessible_with_any_key
9/1000 8e1f9e7d key= not_accessible_with_any_key
10/1000 44346bf3 key= not_accessible_with_any_key
11/1000 e1658161 key= not_accessible_with_any_key
12/1000 aed160da key=2 rejected
13/1000 5be01e77 key= not_accessible_with_any_key
14/1000 a1f32fea key= not_accessible_with_any_key
15/1000 e2007b7d key= not_accessible_with_any_key
16/1000 26a0df13 key= not_accessible_with_any_

In [ ]:
#use this to check the health of your api keys
from pathlib import Path
import csv
import time
import requests

ADS_URL = "https://ads.atmosphere.copernicus.eu/api"
DATASET = "cams-global-reanalysis-eac4"

ADS_KEYS = [
    "156656bf-4346-4228-8ea6-a47b8b807223",
    "1e0d795d-e02a-4ebc-81f9c07f85a9",
    "1e912241-e383-49bc-a605-4f3155b60ad6",
    "9f9d3677-2705-4764-832b-c5d9bc917eb0",
    "9b7c19b2-d4c7-4ab6-9aef-33d57355862a",
    "2d7328f2-7e18-4fb4-ad55-2ca4605eb1c4",
    "22dec1aa-ecf2-4b87-9641-b80361fdba06",
    "d952344a-3e13-45a6-a7cd-4a7b8e220a79",
    "b6a8b25e-6d7f-46f8-80f6-cf0e37a2719b",
    "40b9f769-d2f6-4369-9b1f-15071345c640",
    "71c70660-fc78-41ce-836b-562ad7f3e1bc",
    "27446185-27c3-4c5d-b178-debf0401be69",
    "54964f95-2780-4c66-b9ef-43e8443d1335",
    "9fd1440b-d970-438c-a31d-1b68807111df",
    "1058bf2f-507b-409e-b072-e36c65f5415d",
]

REQUEST = {
    "variable": ["10m_u_component_of_wind"],
    "date": "2021-04-03/2021-04-03",
    "time": ["18:00"],
    "area": [55.7, -11.2, 50.8, -5.2],
    "data_format": "grib",
}

OUTDIR = Path(r"D:/CAMS_key_test_ireland_20210403_18UTC")
OUTDIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = OUTDIR / "ads_key_test_summary_fast.csv"

MAX_WAIT_SECONDS = 35
POLL_SECONDS = 4
HTTP_TIMEOUT = 20

def headers(key):
    return {"PRIVATE-TOKEN": key, "Content-Type": "application/json"}

def execute_url():
    return f"{ADS_URL}/retrieve/v1/processes/{DATASET}/execute/"

def job_url(job_id):
    return f"{ADS_URL}/retrieve/v1/jobs/{job_id}"

def results_url(job_id):
    return f"{ADS_URL}/retrieve/v1/jobs/{job_id}/results"

def classify_http(code, text):
    x = text.lower()
    if code in (401, 403):
        return "BANNED_OR_INVALID"
    if "invalid token" in x or "private-token" in x or "unauthorized" in x or "forbidden" in x:
        return "BANNED_OR_INVALID"
    if "temporarily limited" in x or "queued requests" in x or "rejected" in x:
        return "VALID_BUT_DATASET_QUEUE_LIMITED"
    if code == 429:
        return "RATE_LIMITED"
    if code in (500, 502, 503, 504):
        return "TRANSIENT_SERVER_ERROR"
    if code == 400:
        return "REQUEST_ERROR"
    return f"HTTP_{code}"

def delete_job(key, job_id):
    try:
        requests.delete(job_url(job_id), headers=headers(key), timeout=HTTP_TIMEOUT)
    except Exception:
        pass

def test_key(i, key):
    t0 = time.time()
    job_id = ""
    last_status = ""
    message = ""

    try:
        r = requests.post(
            execute_url(),
            json={"inputs": REQUEST},
            headers=headers(key),
            timeout=HTTP_TIMEOUT,
        )

        if r.status_code >= 400:
            return {
                "key_index": i,
                "key_prefix": key[:8],
                "status": classify_http(r.status_code, r.text),
                "job_id": "",
                "last_server_status": "",
                "elapsed_seconds": round(time.time() - t0, 1),
                "message": r.text[:1000],
            }

        body = r.json()
        job_id = body.get("jobID") or body.get("id") or body.get("request_uid") or ""
        if not job_id:
            return {
                "key_index": i,
                "key_prefix": key[:8],
                "status": "SUBMIT_OK_NO_JOB_ID",
                "job_id": "",
                "last_server_status": "",
                "elapsed_seconds": round(time.time() - t0, 1),
                "message": str(body)[:1000],
            }

        while time.time() - t0 < MAX_WAIT_SECONDS:
            p = requests.get(job_url(job_id), headers=headers(key), timeout=HTTP_TIMEOUT)

            if p.status_code >= 400:
                return {
                    "key_index": i,
                    "key_prefix": key[:8],
                    "status": classify_http(p.status_code, p.text),
                    "job_id": job_id,
                    "last_server_status": last_status,
                    "elapsed_seconds": round(time.time() - t0, 1),
                    "message": p.text[:1000],
                }

            pb = p.json()
            last_status = str(pb.get("status", "")).lower()
            message = str(pb)[:1000]

            if last_status == "successful":
                return {
                    "key_index": i,
                    "key_prefix": key[:8],
                    "status": "VALID_AND_JOB_SUCCESSFUL",
                    "job_id": job_id,
                    "last_server_status": last_status,
                    "elapsed_seconds": round(time.time() - t0, 1),
                    "message": "Key accepted and job completed.",
                }

            if last_status == "rejected":
                return {
                    "key_index": i,
                    "key_prefix": key[:8],
                    "status": "VALID_BUT_DATASET_QUEUE_LIMITED",
                    "job_id": job_id,
                    "last_server_status": last_status,
                    "elapsed_seconds": round(time.time() - t0, 1),
                    "message": message,
                }

            if last_status in {"failed", "dismissed", "deleted"}:
                return {
                    "key_index": i,
                    "key_prefix": key[:8],
                    "status": f"VALID_BUT_JOB_{last_status.upper()}",
                    "job_id": job_id,
                    "last_server_status": last_status,
                    "elapsed_seconds": round(time.time() - t0, 1),
                    "message": message,
                }

            time.sleep(POLL_SECONDS)

        delete_job(key, job_id)

        return {
            "key_index": i,
            "key_prefix": key[:8],
            "status": "VALID_BUT_SLOW_CANCELLED",
            "job_id": job_id,
            "last_server_status": last_status,
            "elapsed_seconds": round(time.time() - t0, 1),
            "message": f"Still {last_status} after {MAX_WAIT_SECONDS}s; cancelled test job.",
        }

    except Exception as e:
        return {
            "key_index": i,
            "key_prefix": key[:8],
            "status": "LOCAL_OR_NETWORK_ERROR",
            "job_id": job_id,
            "last_server_status": last_status,
            "elapsed_seconds": round(time.time() - t0, 1),
            "message": repr(e)[:1000],
        }

rows = []

for i, key in enumerate(ADS_KEYS):
    print(f"\nTesting key {i:02d}: {key[:8]}...")
    row = test_key(i, key)
    rows.append(row)
    print(f"key {i:02d}: {row['status']} | server={row['last_server_status']} | {row['elapsed_seconds']}s")

    with open(SUMMARY_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "key_index",
                "key_prefix",
                "status",
                "job_id",
                "last_server_status",
                "elapsed_seconds",
                "message",
            ],
        )
        writer.writeheader()
        writer.writerows(rows)

    time.sleep(1)

print("\nSummary")
print("=" * 80)

for r in rows:
    print(
        f"key {r['key_index']:02d} {r['key_prefix']} "
        f"{r['status']} "
        f"server={r['last_server_status']} "
        f"{r['elapsed_seconds']}s"
    )

print(f"\nWrote: {SUMMARY_CSV}")